In [7]:
!wget https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip ml-1m.zip


--2025-11-25 00:25:56--  https://files.grouplens.org/datasets/movielens/ml-1m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5917549 (5.6M) [application/zip]
Saving to: ‘ml-1m.zip.1’

ml-1m.zip.1         100%[===================>]   5.64M  25.0MB/s    in 0.2s    

2025-11-25 00:25:56 (25.0 MB/s) - ‘ml-1m.zip.1’ saved [5917549/5917549]

Archive:  ml-1m.zip
replace ml-1m/movies.dat? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-1m/movies.dat        
replace ml-1m/ratings.dat? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-1m/ratings.dat       
replace ml-1m/README? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-1m/README            
replace ml-1m/users.dat? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-1m/users.dat         


In [8]:
import pandas as pd

ratings_path = "ml-1m/ratings.dat"
df = pd.read_csv(ratings_path,
            sep="::",
            engine = "python",
            names=["user_id", "item_id", "rating", "timestamp"])

#Converting ratings to implicit feedback
df["label"] = (df["rating"]>=4).astype(int)
df = df[["user_id","item_id","label"]]

In [9]:
df.head()

,user_id,item_id,label
0,1,1193,1
1,1,661,0
2,1,914,0
3,1,3408,1
4,1,2355,1


In [6]:
df.shape

(1000209, 3)

In [10]:
from sklearn.preprocessing import LabelEncoder
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df["user_id"] = user_encoder.fit_transform(df["user_id"])
df["item_id"] = item_encoder.fit_transform(df["item_id"])

num_users = df["user_id"].nunique()
num_items = df["item_id"].nunique()

num_users, num_items

(6040, 3706)

In [11]:
import numpy as np
df = df.sort_values(["user_id"])

train_data = []
test_data = []

for user, group in df.groupby("user_id"):
  group = group.sort_values("item_id")

  test_row = group.iloc[-1]
  train_row = group.iloc[:-1]

  train_data.append(train_row)
  test_data.append(test_row)

train_df = pd.concat(train_data)
test_df = pd.DataFrame(test_data)

In [12]:
train_df.shape, test_df.shape

((994169, 3), (6040, 3))

In [13]:
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [14]:
user_pos_items = (
    train_df.groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

In [17]:
#This samples items the user has never interacted with.
import random

def sample_negative_items(user, num_negatives, num_items, user_pos_dict):
  negatives = []
  positives = user_pos_dict[user]

  while len(negatives) < num_negatives:
    neg = random.randint(0, num_items-1)
    if neg not in positives:
      negatives.append(neg)

  return negatives

In [18]:
num_neg = 4  # standard

train_users = []
train_items = []
train_labels = []

for row in train_df.itertuples():
    u = row.user_id
    i = row.item_id

    # Add positive sample
    train_users.append(u)
    train_items.append(i)
    train_labels.append(1)

    # Add negative samples
    negs = sample_negative_items(u, num_neg, num_items, user_pos_items)
    for neg in negs:
        train_users.append(u)
        train_items.append(neg)
        train_labels.append(0)


In [19]:
import torch

train_users = torch.tensor(train_users, dtype=torch.long)
train_items = torch.tensor(train_items, dtype=torch.long)
train_labels = torch.tensor(train_labels, dtype=torch.float32)

In [20]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(train_users, train_items, train_labels)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

In [21]:
#Generalized Matrix Factorization
import torch
import torch.nn as nn

class GMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=32):
        super(GMF, self).__init__()

        # Embeddings
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)

        # Prediction layer
        self.output = nn.Linear(latent_dim, 1)

        # Sigmoid for implicit probability
        self.sigmoid = nn.Sigmoid()

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        nn.init.kaiming_uniform_(self.output.weight, a=1, nonlinearity='sigmoid')
        nn.init.zeros_(self.output.bias)

    def forward(self, user_ids, item_ids):
        # Embedding lookup
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)

        # Element-wise multiplication
        x = user_vec * item_vec

        # Predict
        x = self.output(x)

        return self.sigmoid(x).squeeze()


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GMF(num_users=num_users, num_items=num_items, latent_dim=32).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [23]:
from tqdm import tqdm

def train_gmf(model, train_loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0

        for batch_users, batch_items, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch_users = batch_users.to(device)
            batch_items = batch_items.to(device)
            batch_labels = batch_labels.to(device)

            # Forward pass
            preds = model(batch_users, batch_items)

            # Loss
            loss = criterion(preds, batch_labels)
            total_loss += loss.item()

            # Backprop
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")


In [24]:
train_gmf(model, train_loader, epochs=5)

Epoch 1: 100%|██████████| 4855/4855 [01:46<00:00, 45.48it/s]


Epoch 1 Loss: 1791.4886


Epoch 2: 100%|██████████| 4855/4855 [01:43<00:00, 47.11it/s]


Epoch 2 Loss: 1463.3823


Epoch 3: 100%|██████████| 4855/4855 [01:48<00:00, 44.65it/s]


Epoch 3 Loss: 1354.2230


Epoch 4: 100%|██████████| 4855/4855 [01:44<00:00, 46.44it/s]


Epoch 4 Loss: 1271.8786


Epoch 5: 100%|██████████| 4855/4855 [01:47<00:00, 45.16it/s]

Epoch 5 Loss: 1215.9395


In [25]:
import numpy as np

# Number of negatives for evaluation
num_eval_negatives = 99

# Prepare dictionary for test items
test_user_item = dict(zip(test_df['user_id'], test_df['item_id']))

In [26]:
def hit_ratio(ranklist, gt_item):
    return int(gt_item in ranklist)

def ndcg(ranklist, gt_item):
    if gt_item in ranklist:
        index = ranklist.index(gt_item)
        return np.reciprocal(np.log2(index + 2))
    return 0

In [27]:
K = 10
HR, NDCG = [], []

model.eval()
with torch.no_grad():
    for user in test_user_item.keys():
        gt_item = test_user_item[user]
        # Sample 99 negatives
        neg_items = []
        while len(neg_items) < num_eval_negatives:
            neg = np.random.randint(0, num_items)
            if neg not in user_pos_items[user] and neg != gt_item:
                neg_items.append(neg)

        # Combine positive + negatives
        item_candidates = [gt_item] + neg_items
        user_tensor = torch.tensor([user]*len(item_candidates), dtype=torch.long).to(device)
        item_tensor = torch.tensor(item_candidates, dtype=torch.long).to(device)

        # Predict scores
        scores = model(user_tensor, item_tensor).cpu().numpy()

        # Rank items by score descending
        ranklist = [x for _, x in sorted(zip(scores, item_candidates), reverse=True)]

        HR.append(hit_ratio(ranklist[:K], gt_item))
        NDCG.append(ndcg(ranklist[:K], gt_item))

In [28]:
print(f"Hit Ratio @ {K}: {np.mean(HR):.4f}")
print(f"NDCG @ {K}: {np.mean(NDCG):.4f}")


Hit Ratio @ 10: 0.4661
NDCG @ 10: 0.2389


In [29]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, layers=[64,32,16,8]):
        super(MLP, self).__init__()

        # Embeddings
        self.user_embedding = nn.Embedding(num_users, layers[0]//2)
        self.item_embedding = nn.Embedding(num_items, layers[0]//2)

        # MLP layers
        mlp_layers = []
        input_size = layers[0]
        for layer_size in layers[1:]:
            mlp_layers.append(nn.Linear(input_size, layer_size))
            mlp_layers.append(nn.ReLU())
            input_size = layer_size
        self.mlp = nn.Sequential(*mlp_layers)

        # Prediction layer
        self.output = nn.Linear(input_size, 1)
        self.sigmoid = nn.Sigmoid()

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        nn.init.kaiming_uniform_(self.output.weight, a=1, nonlinearity='sigmoid')
        nn.init.zeros_(self.output.bias)

    def forward(self, user_ids, item_ids):
        user_vec = self.user_embedding(user_ids)
        item_vec = self.item_embedding(item_ids)
        x = torch.cat([user_vec, item_vec], dim=-1)
        x = self.mlp(x)
        x = self.output(x)
        return self.sigmoid(x).squeeze()

In [30]:
mlp_model = MLP(num_users=num_users, num_items=num_items).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)


In [31]:
def train_mlp(model, train_loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_users, batch_items, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch_users = batch_users.to(device)
            batch_items = batch_items.to(device)
            batch_labels = batch_labels.to(device)

            preds = model(batch_users, batch_items)
            loss = criterion(preds, batch_labels)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

In [32]:
train_mlp(mlp_model, train_loader, epochs=5)

Epoch 1: 100%|██████████| 4855/4855 [01:52<00:00, 43.10it/s]


Epoch 1 Loss: 1763.2582


Epoch 2: 100%|██████████| 4855/4855 [01:49<00:00, 44.21it/s]


Epoch 2 Loss: 1607.5442


Epoch 3: 100%|██████████| 4855/4855 [01:51<00:00, 43.40it/s]


Epoch 3 Loss: 1517.5958


Epoch 4: 100%|██████████| 4855/4855 [01:51<00:00, 43.69it/s]


Epoch 4 Loss: 1438.4019


Epoch 5: 100%|██████████| 4855/4855 [01:50<00:00, 43.76it/s]

Epoch 5 Loss: 1383.7996


In [33]:
class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, mf_dim=32, layers=[64,32,16,8]):
        super(NeuMF, self).__init__()

        # GMF embeddings
        self.gmf_user_emb = nn.Embedding(num_users, mf_dim)
        self.gmf_item_emb = nn.Embedding(num_items, mf_dim)

        # MLP embeddings
        self.mlp_user_emb = nn.Embedding(num_users, layers[0]//2)
        self.mlp_item_emb = nn.Embedding(num_items, layers[0]//2)

        # MLP layers
        mlp_layers = []
        input_size = layers[0]
        for layer_size in layers[1:]:
            mlp_layers.append(nn.Linear(input_size, layer_size))
            mlp_layers.append(nn.ReLU())
            input_size = layer_size
        self.mlp = nn.Sequential(*mlp_layers)

        # Final prediction layer
        self.output = nn.Linear(mf_dim + input_size, 1)
        self.sigmoid = nn.Sigmoid()

        self._init_weights()

    def _init_weights(self):
        for emb in [self.gmf_user_emb, self.gmf_item_emb,
                    self.mlp_user_emb, self.mlp_item_emb]:
            nn.init.normal_(emb.weight, std=0.01)
        nn.init.kaiming_uniform_(self.output.weight, a=1, nonlinearity='sigmoid')
        nn.init.zeros_(self.output.bias)

    def forward(self, user_ids, item_ids):
        # GMF branch
        gmf_u = self.gmf_user_emb(user_ids)
        gmf_i = self.gmf_item_emb(item_ids)
        gmf_out = gmf_u * gmf_i

        # MLP branch
        mlp_u = self.mlp_user_emb(user_ids)
        mlp_i = self.mlp_item_emb(item_ids)
        mlp_out = torch.cat([mlp_u, mlp_i], dim=-1)
        mlp_out = self.mlp(mlp_out)

        # Concatenate and predict
        final_input = torch.cat([gmf_out, mlp_out], dim=-1)
        x = self.output(final_input)
        return self.sigmoid(x).squeeze()


In [34]:
model = NeuMF(num_users=num_users, num_items=num_items).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_neumf(model, train_loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_users, batch_items, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch_users = batch_users.to(device)
            batch_items = batch_items.to(device)
            batch_labels = batch_labels.to(device)

            preds = model(batch_users, batch_items)
            loss = criterion(preds, batch_labels)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")


In [35]:
train_neumf(model, train_loader, epochs=5)

Epoch 1: 100%|██████████| 4855/4855 [02:06<00:00, 38.49it/s]


Epoch 1 Loss: 1575.1130


Epoch 2: 100%|██████████| 4855/4855 [02:02<00:00, 39.71it/s]


Epoch 2 Loss: 1286.4757


Epoch 3: 100%|██████████| 4855/4855 [02:03<00:00, 39.42it/s]


Epoch 3 Loss: 1185.7178


Epoch 4: 100%|██████████| 4855/4855 [02:05<00:00, 38.73it/s]


Epoch 4 Loss: 1119.9768


Epoch 5: 100%|██████████| 4855/4855 [02:03<00:00, 39.26it/s]

Epoch 5 Loss: 1076.8353


In [36]:
def evaluate_model(model, test_user_item, user_pos_items, K=10, num_items=num_items, device=device):
    HR, NDCG = [], []
    model.eval()

    with torch.no_grad():
        for user, gt_item in test_user_item.items():
            # Sample 99 negatives
            neg_items = []
            while len(neg_items) < 99:
                neg = np.random.randint(0, num_items)
                if neg not in user_pos_items[user] and neg != gt_item:
                    neg_items.append(neg)

            item_candidates = [gt_item] + neg_items
            user_tensor = torch.tensor([user]*len(item_candidates), dtype=torch.long).to(device)
            item_tensor = torch.tensor(item_candidates, dtype=torch.long).to(device)

            # Predict
            scores = model(user_tensor, item_tensor).cpu().numpy()
            # Rank items
            ranklist = [x for _, x in sorted(zip(scores, item_candidates), reverse=True)]

            # Metrics
            HR.append(int(gt_item in ranklist[:K]))
            if gt_item in ranklist[:K]:
                index = ranklist.index(gt_item)
                NDCG.append(1 / np.log2(index + 2))
            else:
                NDCG.append(0)

    return np.mean(HR), np.mean(NDCG)


In [37]:
mlp_hr, mlp_ndcg = evaluate_model(mlp_model, test_user_item, user_pos_items, K=10)
print(f"MLP — HR@10: {mlp_hr:.4f}, NDCG@10: {mlp_ndcg:.4f}")

MLP — HR@10: 0.4101, NDCG@10: 0.2060


In [40]:
neumf_hr, neumf_ndcg = evaluate_model(model, test_user_item, user_pos_items, K=10)
print(f"NeuMF — HR@10: {neumf_hr:.4f}, NDCG@10: {neumf_ndcg:.4f}")

NeuMF — HR@10: 0.5030, NDCG@10: 0.2660


In [49]:
torch.save(model.state_dict(), "neumf.pth")

In [52]:
def recommend_for_user(model, user_id, top_k=10, num_items=num_items):
    model.eval()
    with torch.no_grad():
        user_tensor = torch.tensor([user_id]*num_items)
        item_tensor = torch.arange(num_items)
        scores = model(user_tensor, item_tensor).squeeze()
        top_items = torch.topk(scores, top_k).indices.tolist()
    return top_items


In [58]:
top_items = recommend_for_user(model, user_id=5, top_k=10)
print(top_items)

[970, 1899, 1900, 853, 1215, 1107, 1281, 838, 1767, 858]
